In [3]:
import json
import os
import requests


print(f"Libraries 'json', 'os', and 'requests' imported. File paths defined:")

Libraries 'json', 'os', and 'requests' imported. File paths defined:


In [4]:
# SECTION: Input Logic App as a text block

import os

input_path = globals().get("input_path", "/content/input_logic_app.json")

with open(input_path, "r", encoding="utf-8") as f:
    input_logic_app_content = f.read()

In [5]:
# SECTION: ARM SCHEMA JSON
arm_schema_path = '''
{
  "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentTemplate.json#",
  "contentVersion": "1.0.0.0",
  "metadata": {
    "title": "Logic app Title",
    "description": "Logic app description",
    "prerequisites": "",
    "lastUpdateTime": "2026-03-06T00:00:00.000Z",
    "author": {
      "name": "Abhi Patidar"
    }
  },

  "parameters": {
    "PlaybookName": {
      "defaultValue": "Logic app name",
      "type": "String"
    }
  },

  "variables": {
    "azureMonitorConnectionName": "[concat('azuremonitorlogs-', parameters('PlaybookName'))]"
  },

  "resources": [

    {
  "type": "Microsoft.Web/connections",
  "apiVersion": "2016-06-01",
  "name": "[variables('azureMonitorConnectionName')]",
  "location": "[resourceGroup().location]",
  "properties": {
    "displayName": "[parameters('PlaybookName')]",
    "api": {
      "id": "[concat('/subscriptions/', subscription().subscriptionId, '/providers/Microsoft.Web/locations/', resourceGroup().location, '/managedApis/azuremonitorlogs')]"
    }
  }
},



    {
      "type": "Microsoft.Logic/workflows",
      "apiVersion": "2017-07-01",
      "name": "[parameters('PlaybookName')]",
      "location": "[resourceGroup().location]",
      "identity": {
        "type": "SystemAssigned"
      },
      "dependsOn": [
        "[resourceId('Microsoft.Web/connections', variables('azureMonitorConnectionName'))]"
      ],
      "properties": {
        "state": "Enabled",
        "definition": {
          "$schema": "https://schema.management.azure.com/providers/Microsoft.Logic/schemas/2016-06-01/workflowdefinition.json#",
          "contentVersion": "1.0.0.0",
          "parameters": {
            "$connections": {
              "defaultValue": {},
              "type": "Object"
            }
          },

          "triggers": {
          },

          "actions": {
          },

          "outputs": {}
        },

        "parameters": {
          "$connections": {
            "value": {
              "azuremonitorlogs": {
                "connectionId": "[resourceId('Microsoft.Web/connections', variables('azureMonitorConnectionName'))]",
  "connectionName": "[variables('azureMonitorConnectionName')]",
  "id": "[concat('/subscriptions/', subscription().subscriptionId, '/providers/Microsoft.Web/locations/', resourceGroup().location, '/managedApis/azuremonitorlogs')]"
              }
            }
          }
        }
      }
    }
  ]

}
'''

print("ARM SCHEMA Logic App content defined as a string block.")

ARM SCHEMA Logic App content defined as a string block.


In [6]:
# SECTION: ORIGINAL LOGIC APP

original_logic_app_path = '''
{
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentTemplate.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "workflows_Izoolab_Incident_Ingestion_name": {
            "defaultValue": "Izoolab_Incident_Ingestion",
            "type": "String"
        },
        "connections_azuremonitorlogs_4_externalid": {
            "defaultValue": "/subscriptions/016d4828-4759-41cd-b6c7-bd973332fa27/resourceGroups/PCI_SEC_RG/providers/Microsoft.Web/connections/azuremonitorlogs-4",
            "type": "String"
        },
        "connections_azureloganalyticsdatacollector_externalid": {
            "defaultValue": "/subscriptions/016d4828-4759-41cd-b6c7-bd973332fa27/resourceGroups/PCI_SEC_RG/providers/Microsoft.Web/connections/azureloganalyticsdatacollector",
            "type": "String"
        }
    },
    "variables": {},
    "resources": [
        {
            "type": "Microsoft.Logic/workflows",
            "apiVersion": "2017-07-01",
            "name": "[parameters('workflows_Izoolab_Incident_Ingestion_name')]",
            "location": "centralindia",
            "identity": {
                "type": "SystemAssigned"
            },
            "properties": {
                "state": "Enabled",
                "definition": {
                    "$schema": "https://schema.management.azure.com/providers/Microsoft.Logic/schemas/2016-06-01/workflowdefinition.json#",
                    "contentVersion": "1.0.0.0",
                    "parameters": {
                        "$connections": {
                            "defaultValue": {},
                            "type": "Object"
                        }
                    },
                    "triggers": {
                        "Recurrence": {
                            "recurrence": {
                                "interval": 30,
                                "frequency": "Minute"
                            },
                            "evaluatedRecurrence": {
                                "interval": 30,
                                "frequency": "Minute"
                            },
                            "type": "Recurrence"
                        }
                    },
                    "actions": {
                        "HTTP_Auth": {
                            "runAfter": {
                                "Initialize_variables": [
                                    "Succeeded"
                                ]
                            },
                            "type": "Http",
                            "inputs": {
                                "uri": "https://client-api.izoolabs.com/api/Token/Authenticate",
                                "method": "POST",
                                "headers": {
                                    "Content-Type": "application/json"
                                },
                                "body": {
                                    "apiKey": "93171A4A-1636-4295-9508-7619A3C77C30",
                                    "secretKey": "1D5C1619-10BD-4474-B649-128E1A0CE9D450EE17BA-BDEC-4AFF-8225-8CDD681EF85B"
                                }
                            },
                            "runtimeConfiguration": {
                                "contentTransfer": {
                                    "transferMode": "Chunked"
                                }
                            }
                        },
                        "HTTP_Incident": {
                            "runAfter": {
                                "HTTP_Auth": [
                                    "Succeeded"
                                ]
                            },
                            "type": "Http",
                            "inputs": {
                                "uri": "https://client-api.izoolabs.com/api/ThreatManagement/FetchIncidents",
                                "method": "POST",
                                "headers": {
                                    "Authorization": "Bearer @{body('HTTP_Auth')?['result']?['accessToken']}"
                                },
                                "body": {
                                    "fromDate": "@{div(sub(ticks(addMinutes(utcNow(),-31)),ticks('1970-01-01T00:00:00Z')),10000000)\n}",
                                    "toDate": "@{div(sub(ticks(utcNow()),ticks('1970-01-01T00:00:00Z')),10000000)}"
                                }
                            },
                            "runtimeConfiguration": {
                                "contentTransfer": {
                                    "transferMode": "Chunked"
                                }
                            }
                        },
                        "Initialize_variables": {
                            "runAfter": {},
                            "type": "InitializeVariable",
                            "inputs": {
                                "variables": [
                                    {
                                        "name": "logs",
                                        "type": "array"
                                    },
                                    {
                                        "name": "files",
                                        "type": "array"
                                    }
                                ]
                            }
                        },
                        "if_no_Incident": {
                            "actions": {},
                            "runAfter": {
                                "HTTP_Incident": [
                                    "Succeeded"
                                ]
                            },
                            "else": {
                                "actions": {
                                    "For_each_Incident": {
                                        "foreach": "@body('HTTP_Incident')?['result']?['incidents']",
                                        "actions": {
                                            "check_incident_type": {
                                                "actions": {
                                                    "Run_query_and_list_results": {
                                                        "type": "ApiConnection",
                                                        "inputs": {
                                                            "host": {
                                                                "connection": {
                                                                    "name": "@parameters('$connections')['azuremonitorlogs-1']['connectionId']"
                                                                }
                                                            },
                                                            "method": "post",
                                                            "body": "izoolab_CL\n| where incidentID_s == \"@{items('For_each_Incident')?['incidentID']}\"",
                                                            "path": "/queryData",
                                                            "queries": {
                                                                "subscriptions": "016d4828-4759-41cd-b6c7-bd973332fa27",
                                                                "resourcegroups": "PCI_SEC_RG",
                                                                "resourcetype": "Log Analytics Workspace",
                                                                "resourcename": "PineLabsSentinelProd",
                                                                "timerange": "7d"
                                                            }
                                                        }
                                                    },
                                                    "Condition_2": {
                                                        "actions": {},
                                                        "runAfter": {
                                                            "Run_query_and_list_results": [
                                                                "Succeeded"
                                                            ]
                                                        },
                                                        "else": {
                                                            "actions": {
                                                                "left_data": {
                                                                    "type": "Compose",
                                                                    "inputs": "@items('For_each_Incident')"
                                                                },
                                                                "HTTP_case_details": {
                                                                    "runAfter": {
                                                                        "left_data": [
                                                                            "Succeeded"
                                                                        ]
                                                                    },
                                                                    "type": "Http",
                                                                    "inputs": {
                                                                        "uri": "https://client-api.izoolabs.com/api/ThreatManagement/CaseDetail?caseId=@{items('For_each_Incident')?['incidentID']}",
                                                                        "method": "GET",
                                                                        "headers": {
                                                                            "Authorization": "Bearer @{body('HTTP_Auth')?['result']?['accessToken']}"
                                                                        }
                                                                    },
                                                                    "runtimeConfiguration": {
                                                                        "contentTransfer": {
                                                                            "transferMode": "Chunked"
                                                                        }
                                                                    }
                                                                },
                                                                "right_data": {
                                                                    "runAfter": {
                                                                        "HTTP_case_details": [
                                                                            "Succeeded"
                                                                        ]
                                                                    },
                                                                    "type": "Compose",
                                                                    "inputs": "@body('HTTP_case_details')?['result']"
                                                                },
                                                                "Output": {
                                                                    "runAfter": {
                                                                        "right_data": [
                                                                            "Succeeded"
                                                                        ]
                                                                    },
                                                                    "type": "Compose",
                                                                    "inputs": "@union(outputs('left_data'), outputs('right_data'))\n"
                                                                },
                                                                "top": {
                                                                    "runAfter": {
                                                                        "Output": [
                                                                            "Succeeded"
                                                                        ]
                                                                    },
                                                                    "type": "Compose",
                                                                    "inputs": {
                                                                        "incidentID": "@{outputs('output')?['incidentID']}",
                                                                        "brandCode": "@{outputs('output')?['brandCode']}",
                                                                        "brand": "@{outputs('output')?['brand']}",
                                                                        "brandId": "@{outputs('output')?['brandId']}",
                                                                        "url": "@{outputs('output')?['url']}",
                                                                        "incidentType": "@{outputs('output')?['incidentType']}",
                                                                        "subIncidentType": "@{outputs('output')?['subIncidentType']}",
                                                                        "threatType": "@{outputs('output')?['threatType']}",
                                                                        "status": "@{outputs('output')?['status']}",
                                                                        "statusCode": "@{outputs('output')?['statusCode']}",
                                                                        "clientRef": "@{outputs('output')?['clientRef']}",
                                                                        "createdOn": "@{if(empty(outputs('output')?['createdOn']), null, formatDateTime(addSeconds('1970-01-01T00:00:00Z', int(outputs('output')?['createdOn'])), 'yyyy-MM-ddTHH:mm:ssZ'))}",
                                                                        "closedOn": "@{if(empty(outputs('output')?['closedOn']), null, formatDateTime(addSeconds('1970-01-01T00:00:00Z', int(outputs('output')?['closedOn'])), 'yyyy-MM-ddTHH:mm:ssZ'))}",
                                                                        "detectionDate": "@{if(empty(outputs('output')?['detectionDate']), null, formatDateTime(addSeconds('1970-01-01T00:00:00Z', int(outputs('output')?['detectionDate'])), 'yyyy-MM-ddTHH:mm:ssZ'))}",
                                                                        "addedDate": "@{if(empty(outputs('output')?['addedDate']), null, formatDateTime(addSeconds('1970-01-01T00:00:00Z', int(outputs('output')?['addedDate'])), 'yyyy-MM-ddTHH:mm:ssZ'))}",
                                                                        "uptime": "@{outputs('output')?['uptime']}",
                                                                        "detectedBy": "@{outputs('output')?['detectedBy']}",
                                                                        "executiveName": "@{outputs('output')?['executiveName']}",
                                                                        "caseType": "@{outputs('output')?['caseType']}",
                                                                        "caseDescription": "@{outputs('output')?['caseDescription']}",
                                                                        "statusDescription": "@{outputs('output')?['statusDescription']}",
                                                                        "description": "@{outputs('output')?['description']}",
                                                                        "impact": "@{outputs('output')?['impact']}",
                                                                        "recommendation": "@{outputs('output')?['recommendation']}",
                                                                        "ipAddress": "@{outputs('output')?['ipAddress']}",
                                                                        "webhost": "@{outputs('output')?['webhost']}",
                                                                        "webhostCountry": "@{outputs('output')?['webhostCountry']}",
                                                                        "registrarName": "@{outputs('output')?['registrarName']}",
                                                                        "nameServer": "@{outputs('output')?['nameServer']}",
                                                                        "mxRecord": "@{outputs('output')?['mxRecord']}",
                                                                        "progressStage": "@{outputs('output')?['progressStage']}",
                                                                        "mobileAppName": "@{outputs('output')?['mobileAppName']}",
                                                                        "mobileAppVersion": "@{outputs('output')?['mobileAppVersion']}",
                                                                        "mobileAppStatus": "@{outputs('output')?['mobileAppStatus']}",
                                                                        "mobileAppSignature": "@{outputs('output')?['mobileAppSignature']}"
                                                                    }
                                                                },
                                                                "final_output": {
                                                                    "runAfter": {
                                                                        "Condition_1": [
                                                                            "Succeeded"
                                                                        ]
                                                                    },
                                                                    "type": "Compose",
                                                                    "inputs": "@setProperty(\n    setProperty(\n        outputs('top'),\n        'incidentStatusLogs',\n        variables('logs')\n    ),\n    'caseFiles',\n    variables('files')\n)\n"
                                                                },
                                                                "Send_Data": {
                                                                    "runAfter": {
                                                                        "final_output": [
                                                                            "Succeeded"
                                                                        ]
                                                                    },
                                                                    "type": "ApiConnection",
                                                                    "inputs": {
                                                                        "host": {
                                                                            "connection": {
                                                                                "name": "@parameters('$connections')['azureloganalyticsdatacollector']['connectionId']"
                                                                            }
                                                                        },
                                                                        "method": "post",
                                                                        "body": "@{outputs('final_output')}",
                                                                        "headers": {
                                                                            "Log-Type": "izoolab_CL"
                                                                        },
                                                                        "path": "/api/logs"
                                                                    }
                                                                },
                                                                "Condition": {
                                                                    "actions": {},
                                                                    "runAfter": {
                                                                        "top": [
                                                                            "Succeeded"
                                                                        ]
                                                                    },
                                                                    "else": {
                                                                        "actions": {
                                                                            "For_each": {
                                                                                "foreach": "@outputs('Output')?['incidentStatusLogs']",
                                                                                "actions": {
                                                                                    "logs_append": {
                                                                                        "type": "AppendToArrayVariable",
                                                                                        "inputs": {
                                                                                            "name": "logs",
                                                                                            "value": {
                                                                                                "incidentStatus": "@items('For_each')?['incidentStatus']",
                                                                                                "incidentStatusDescription": "@{string(items('For_each')?['incidentStatusDescription'])}",
                                                                                                "remark": "@{string(items('For_each')?['remark'])}",
                                                                                                "updatedOn": "@{formatDateTime(convertTimeZone(addSeconds('1970-01-01T00:00:00Z', int(items('For_each')?['updatedOn'])), 'UTC', 'India Standard Time'), 'yyyy-MM-dd HH:mm:ss')}",
                                                                                                "updatedBy": "@{string(items('For_each')?['updatedBy'])}"
                                                                                            }
                                                                                        }
                                                                                    }
                                                                                },
                                                                                "type": "Foreach"
                                                                            }
                                                                        }
                                                                    },
                                                                    "expression": {
                                                                        "and": [
                                                                            {
                                                                                "equals": [
                                                                                    "@outputs('Output')?['incidentStatusLogs']",
                                                                                    "@null"
                                                                                ]
                                                                            }
                                                                        ]
                                                                    },
                                                                    "type": "If"
                                                                },
                                                                "Condition_1": {
                                                                    "actions": {},
                                                                    "runAfter": {
                                                                        "Condition": [
                                                                            "Succeeded"
                                                                        ]
                                                                    },
                                                                    "else": {
                                                                        "actions": {
                                                                            "For_each-copy": {
                                                                                "foreach": "@outputs('Output')?['caseFiles']",
                                                                                "actions": {
                                                                                    "Append_files": {
                                                                                        "type": "AppendToArrayVariable",
                                                                                        "inputs": {
                                                                                            "name": "files",
                                                                                            "value": {
                                                                                                "dateUploaded": "@{formatDateTime(addSeconds('1970-01-01T00:00:00Z', int(items('For_each-copy')?['dateUploaded'])), 'yyyy-MM-dd HH:mm:ss')}",
                                                                                                "originalFilename": "@{string(items('For_each-copy')?['originalFilename'])}",
                                                                                                "fileToken": "@{string(items('For_each-copy')?['fileToken'])}",
                                                                                                "isCaseScreenshot": "@{string(items('For_each-copy')?['isCaseScreenshot'])}"
                                                                                            }
                                                                                        }
                                                                                    }
                                                                                },
                                                                                "type": "Foreach"
                                                                            }
                                                                        }
                                                                    },
                                                                    "expression": {
                                                                        "and": [
                                                                            {
                                                                                "equals": [
                                                                                    "@outputs('Output')?['caseFiles']",
                                                                                    "@null"
                                                                                ]
                                                                            }
                                                                        ]
                                                                    },
                                                                    "type": "If"
                                                                }
                                                            }
                                                        },
                                                        "expression": {
                                                            "and": [
                                                                {
                                                                    "greater": [
                                                                        "@length(body('Run_query_and_list_results')?['value'])",
                                                                        0
                                                                    ]
                                                                }
                                                            ]
                                                        },
                                                        "type": "If"
                                                    }
                                                },
                                                "else": {
                                                    "actions": {}
                                                },
                                                "expression": {
                                                    "or": [
                                                        {
                                                            "equals": [
                                                                "@items('For_each_Incident')?['threatType']",
                                                                "Critical Threat"
                                                            ]
                                                        },
                                                        {
                                                            "equals": [
                                                                "@items('For_each_Incident')?['threatType']",
                                                                "High Threat"
                                                            ]
                                                        },
                                                        {
                                                            "equals": [
                                                                "@items('For_each_Incident')?['threatType']",
                                                                "Substantial Threat"
                                                            ]
                                                        }
                                                    ]
                                                },
                                                "type": "If"
                                            }
                                        },
                                        "type": "Foreach"
                                    }
                                }
                            },
                            "expression": {
                                "and": [
                                    {
                                        "equals": [
                                            "@body('HTTP_Incident')?['result']",
                                            "@null"
                                        ]
                                    },
                                    {
                                        "equals": [
                                            "",
                                            ""
                                        ]
                                    }
                                ]
                            },
                            "type": "If"
                        }
                    },
                    "outputs": {}
                },
                "parameters": {
                    "$connections": {
                        "value": {
                            "azuremonitorlogs-1": {
                                "id": "/subscriptions/016d4828-4759-41cd-b6c7-bd973332fa27/providers/Microsoft.Web/locations/centralindia/managedApis/azuremonitorlogs",
                                "connectionId": "[parameters('connections_azuremonitorlogs_4_externalid')]",
                                "connectionName": "azuremonitorlogs-4",
                                "connectionProperties": {}
                            },
                            "azureloganalyticsdatacollector": {
                                "id": "/subscriptions/016d4828-4759-41cd-b6c7-bd973332fa27/providers/Microsoft.Web/locations/centralindia/managedApis/azureloganalyticsdatacollector",
                                "connectionId": "[parameters('connections_azureloganalyticsdatacollector_externalid')]",
                                "connectionName": "azureloganalyticsdatacollector",
                                "connectionProperties": {}
                            }
                        }
                    }
                }
            }
        }
    ]
}
'''

print("ORIGINAL LOGIC APP content defined as a string block.")

ORIGINAL LOGIC APP content defined as a string block.


In [7]:
# SECTION: PARAMETERISE LOGIC APP

parameterise_logic_app_path = '''
{
  "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentTemplate.json#",
  "contentVersion": "1.0.0.0",
  "metadata": {
    "title": "Izoolabs Incident Sync",
    "description": "This playbook fetches incidents from Izoolabs API, enriches them, checks duplicates in Log Analytics, and ingests into Sentinel custom table.",
    "prerequisites": "Managed Identity must have access to Log Analytics workspace.",
    "lastUpdateTime": "2026-02-23T00:00:00.000Z",
    "author": {
      "name": "Your Name"
    }
  },

  "parameters": {
    "PlaybookName": {
      "defaultValue": "Izoolab_Incident_Ingestion",
      "type": "String"
    },
    "WorkspaceName": {
        "defaultValue": "Izoolab_Workspace",
      "type": "String"
    },
    "WorkspaceResourceGroup": {
        "defaultValue": "Izoolab_Resource_Group",
      "type": "String"
    },
    "QueryTimeRange": {
        "type": "String",
        "defaultValue": "7d"
    }
  },

  "variables": {
    "azureMonitorConnectionName": "[concat('azuremonitorlogs-', parameters('PlaybookName'))]",
    "dataCollectorConnectionName": "[concat('azureloganalyticsdatacollector-', parameters('PlaybookName'))]"
  },

  "resources": [

    {
  "type": "Microsoft.Web/connections",
  "apiVersion": "2016-06-01",
  "name": "[variables('azureMonitorConnectionName')]",
  "location": "[resourceGroup().location]",
  "properties": {
    "displayName": "[parameters('PlaybookName')]",
    "api": {
      "id": "[concat('/subscriptions/', subscription().subscriptionId, '/providers/Microsoft.Web/locations/', resourceGroup().location, '/managedApis/azuremonitorlogs')]"
    }
  }
},



    {
      "type": "Microsoft.Web/connections",
      "apiVersion": "2016-06-01",
      "name": "[variables('dataCollectorConnectionName')]",
      "location": "[resourceGroup().location]",
      "properties": {
        "displayName": "[parameters('PlaybookName')]",
        "api": {
          "id": "[concat('/subscriptions/', subscription().subscriptionId, '/providers/Microsoft.Web/locations/', resourceGroup().location, '/managedApis/azureloganalyticsdatacollector')]"
        }
      }
    },

    {
      "type": "Microsoft.Logic/workflows",
      "apiVersion": "2017-07-01",
      "name": "[parameters('PlaybookName')]",
      "location": "[resourceGroup().location]",
      "identity": {
        "type": "SystemAssigned"
      },
      "dependsOn": [
        "[resourceId('Microsoft.Web/connections', variables('azureMonitorConnectionName'))]",
        "[resourceId('Microsoft.Web/connections', variables('dataCollectorConnectionName'))]"
      ],
      "properties": {
        "state": "Enabled",
        "definition": {
          "$schema": "https://schema.management.azure.com/providers/Microsoft.Logic/schemas/2016-06-01/workflowdefinition.json#",
          "contentVersion": "1.0.0.0",
          "parameters": {
            "$connections": {
              "defaultValue": {},
              "type": "Object"
            }
          },

          "triggers": {
            "Recurrence": {
              "type": "Recurrence",
              "recurrence": {
                "interval": 30,
                "frequency": "Minute"
              }
            }
          },

          "actions": {

                        "HTTP_Auth": {
                            "runAfter": {
                                "Initialize_variables": [
                                    "Succeeded"
                                ]
                            },
                            "type": "Http",
                            "inputs": {
                                "uri": "https://client-api.izoolabs.com/api/Token/Authenticate",
                                "method": "POST",
                                "headers": {
                                    "Content-Type": "application/json"
                                },
                                "body": {
                                    "apiKey": "INSERT API KEY HERE",
                                    "secretKey": "INSERT SECRET KEY HERE"
                                }
                            },
                            "runtimeConfiguration": {
                                "contentTransfer": {
                                    "transferMode": "Chunked"
                                }
                            }
                        },
                        "HTTP_Incident": {
                            "runAfter": {
                                "HTTP_Auth": [
                                    "Succeeded"
                                ]
                            },
                            "type": "Http",
                            "inputs": {
                                "uri": "https://client-api.izoolabs.com/api/ThreatManagement/FetchIncidents",
                                "method": "POST",
                                "headers": {
                                    "Authorization": "Bearer @{body('HTTP_Auth')?['result']?['accessToken']}"
                                },
                                "body": {
                                    "fromDate": "@{div(sub(ticks(addMinutes(utcNow(),-31)),ticks('1970-01-01T00:00:00Z')),10000000)\n}",
                                    "toDate": "@{div(sub(ticks(utcNow()),ticks('1970-01-01T00:00:00Z')),10000000)}"
                                }
                            },
                            "runtimeConfiguration": {
                                "contentTransfer": {
                                    "transferMode": "Chunked"
                                }
                            }
                        },
                        "Initialize_variables": {
                            "runAfter": {},
                            "type": "InitializeVariable",
                            "inputs": {
                                "variables": [
                                    {
                                        "name": "logs",
                                        "type": "array"
                                    },
                                    {
                                        "name": "files",
                                        "type": "array"
                                    }
                                ]
                            }
                        },
                        "if_no_Incident": {
                            "actions": {},
                            "runAfter": {
                                "HTTP_Incident": [
                                    "Succeeded"
                                ]
                            },
                            "else": {
                                "actions": {
                                    "For_each_Incident": {
                                        "foreach": "@body('HTTP_Incident')?['result']?['incidents']",
                                        "actions": {
                                            "check_incident_type": {
                                                "actions": {
                                                    "Run_query_and_list_results": {
                                                        "type": "ApiConnection",
                                                        "inputs": {
                                                            "host": {
                                                                "connection": {
                                                                    "name": "@parameters('$connections')['azuremonitorlogs']['connectionId']"
                                                                }
                                                            },
                                                            "method": "post",
                                                            "body": "izoolab_CL\n| where incidentID_s == \"@{items('For_each_Incident')?['incidentID']}\"",
                                                            "path": "/queryData",
                                                           "queries": {
                                                                "subscriptions": "@{subscription().subscriptionId}",
                                                                "resourcegroups": "@{parameters('WorkspaceResourceGroup')}",
                                                                "resourcetype": "Log Analytics Workspace",
                                                                "resourcename": "@{parameters('WorkspaceName')}",
                                                                "timerange": "@{parameters('QueryTimeRange')}"
                                                            }
                                                        }
                                                    },
                                                    "Condition_2": {
                                                        "actions": {},
                                                        "runAfter": {
                                                            "Run_query_and_list_results": [
                                                                "Succeeded"
                                                            ]
                                                        },
                                                        "else": {
                                                            "actions": {
                                                                "left_data": {
                                                                    "type": "Compose",
                                                                    "inputs": "@items('For_each_Incident')"
                                                                },
                                                                "HTTP_case_details": {
                                                                    "runAfter": {
                                                                        "left_data": [
                                                                            "Succeeded"
                                                                        ]
                                                                    },
                                                                    "type": "Http",
                                                                    "inputs": {
                                                                        "uri": "https://client-api.izoolabs.com/api/ThreatManagement/CaseDetail?caseId=@{items('For_each_Incident')?['incidentID']}",
                                                                        "method": "GET",
                                                                        "headers": {
                                                                            "Authorization": "Bearer @{body('HTTP_Auth')?['result']?['accessToken']}"
                                                                        }
                                                                    },
                                                                    "runtimeConfiguration": {
                                                                        "contentTransfer": {
                                                                            "transferMode": "Chunked"
                                                                        }
                                                                    }
                                                                },
                                                                "right_data": {
                                                                    "runAfter": {
                                                                        "HTTP_case_details": [
                                                                            "Succeeded"
                                                                        ]
                                                                    },
                                                                    "type": "Compose",
                                                                    "inputs": "@body('HTTP_case_details')?['result']"
                                                                },
                                                                "Output": {
                                                                    "runAfter": {
                                                                        "right_data": [
                                                                            "Succeeded"
                                                                        ]
                                                                    },
                                                                    "type": "Compose",
                                                                    "inputs": "@union(outputs('left_data'), outputs('right_data'))\n"
                                                                },
                                                                "top": {
                                                                    "runAfter": {
                                                                        "Output": [
                                                                            "Succeeded"
                                                                        ]
                                                                    },
                                                                    "type": "Compose",
                                                                    "inputs": {
                                                                        "incidentID": "@{outputs('output')?['incidentID']}",
                                                                        "brandCode": "@{outputs('output')?['brandCode']}",
                                                                        "brand": "@{outputs('output')?['brand']}",
                                                                        "brandId": "@{outputs('output')?['brandId']}",
                                                                        "url": "@{outputs('output')?['url']}",
                                                                        "incidentType": "@{outputs('output')?['incidentType']}",
                                                                        "subIncidentType": "@{outputs('output')?['subIncidentType']}",
                                                                        "threatType": "@{outputs('output')?['threatType']}",
                                                                        "status": "@{outputs('output')?['status']}",
                                                                        "statusCode": "@{outputs('output')?['statusCode']}",
                                                                        "clientRef": "@{outputs('output')?['clientRef']}",
                                                                        "createdOn": "@{if(empty(outputs('output')?['createdOn']), null, formatDateTime(addSeconds('1970-01-01T00:00:00Z', int(outputs('output')?['createdOn'])), 'yyyy-MM-ddTHH:mm:ssZ'))}",
                                                                        "closedOn": "@{if(empty(outputs('output')?['closedOn']), null, formatDateTime(addSeconds('1970-01-01T00:00:00Z', int(outputs('output')?['closedOn'])), 'yyyy-MM-ddTHH:mm:ssZ'))}",
                                                                        "detectionDate": "@{if(empty(outputs('output')?['detectionDate']), null, formatDateTime(addSeconds('1970-01-01T00:00:00Z', int(outputs('output')?['detectionDate'])), 'yyyy-MM-ddTHH:mm:ssZ'))}",
                                                                        "addedDate": "@{if(empty(outputs('output')?['addedDate']), null, formatDateTime(addSeconds('1970-01-01T00:00:00Z', int(outputs('output')?['addedDate'])), 'yyyy-MM-ddTHH:mm:ssZ'))}",
                                                                        "uptime": "@{outputs('output')?['uptime']}",
                                                                        "detectedBy": "@{outputs('output')?['detectedBy']}",
                                                                        "executiveName": "@{outputs('output')?['executiveName']}",
                                                                        "caseType": "@{outputs('output')?['caseType']}",
                                                                        "caseDescription": "@{outputs('output')?['caseDescription']}",
                                                                        "statusDescription": "@{outputs('output')?['statusDescription']}",
                                                                        "description": "@{outputs('output')?['description']}",
                                                                        "impact": "@{outputs('output')?['impact']}",
                                                                        "recommendation": "@{outputs('output')?['recommendation']}",
                                                                        "ipAddress": "@{outputs('output')?['ipAddress']}",
                                                                        "webhost": "@{outputs('output')?['webhost']}",
                                                                        "webhostCountry": "@{outputs('output')?['webhostCountry']}",
                                                                        "registrarName": "@{outputs('output')?['registrarName']}",
                                                                        "nameServer": "@{outputs('output')?['nameServer']}",
                                                                        "mxRecord": "@{outputs('output')?['mxRecord']}",
                                                                        "progressStage": "@{outputs('output')?['progressStage']}",
                                                                        "mobileAppName": "@{outputs('output')?['mobileAppName']}",
                                                                        "mobileAppVersion": "@{outputs('output')?['mobileAppVersion']}",
                                                                        "mobileAppStatus": "@{outputs('output')?['mobileAppStatus']}",
                                                                        "mobileAppSignature": "@{outputs('output')?['mobileAppSignature']}"
                                                                    }
                                                                },
                                                                "final_output": {
                                                                    "runAfter": {
                                                                        "Condition_1": [
                                                                            "Succeeded"
                                                                        ]
                                                                    },
                                                                    "type": "Compose",
                                                                    "inputs": "@setProperty(\n    setProperty(\n        outputs('top'),\n        'incidentStatusLogs',\n        variables('logs')\n    ),\n    'caseFiles',\n    variables('files')\n)\n"
                                                                },
                                                                "Send_Data": {
                                                                    "runAfter": {
                                                                        "final_output": [
                                                                            "Succeeded"
                                                                        ]
                                                                    },
                                                                    "type": "ApiConnection",
                                                                    "inputs": {
                                                                        "host": {
                                                                            "connection": {
                                                                                "name": "@parameters('$connections')['azureloganalyticsdatacollector']['connectionId']"
                                                                            }
                                                                        },
                                                                        "method": "post",
                                                                        "body": "@{outputs('final_output')}",
                                                                        "headers": {
                                                                            "Log-Type": "izoolab_CL"
                                                                        },
                                                                        "path": "/api/logs"
                                                                    }
                                                                },
                                                                "Condition": {
                                                                    "actions": {},
                                                                    "runAfter": {
                                                                        "top": [
                                                                            "Succeeded"
                                                                        ]
                                                                    },
                                                                    "else": {
                                                                        "actions": {
                                                                            "For_each": {
                                                                                "foreach": "@outputs('Output')?['incidentStatusLogs']",
                                                                                "actions": {
                                                                                    "logs_append": {
                                                                                        "type": "AppendToArrayVariable",
                                                                                        "inputs": {
                                                                                            "name": "logs",
                                                                                            "value": {
                                                                                                "incidentStatus": "@items('For_each')?['incidentStatus']",
                                                                                                "incidentStatusDescription": "@{string(items('For_each')?['incidentStatusDescription'])}",
                                                                                                "remark": "@{string(items('For_each')?['remark'])}",
                                                                                                "updatedOn": "@{formatDateTime(convertTimeZone(addSeconds('1970-01-01T00:00:00Z', int(items('For_each')?['updatedOn'])), 'UTC', 'India Standard Time'), 'yyyy-MM-dd HH:mm:ss')}",
                                                                                                "updatedBy": "@{string(items('For_each')?['updatedBy'])}"
                                                                                            }
                                                                                        }
                                                                                    }
                                                                                },
                                                                                "type": "Foreach"
                                                                            }
                                                                        }
                                                                    },
                                                                    "expression": {
                                                                        "and": [
                                                                            {
                                                                                "equals": [
                                                                                    "@outputs('Output')?['incidentStatusLogs']",
                                                                                    "@null"
                                                                                ]
                                                                            }
                                                                        ]
                                                                    },
                                                                    "type": "If"
                                                                },
                                                                "Condition_1": {
                                                                    "actions": {},
                                                                    "runAfter": {
                                                                        "Condition": [
                                                                            "Succeeded"
                                                                        ]
                                                                    },
                                                                    "else": {
                                                                        "actions": {
                                                                            "For_each-copy": {
                                                                                "foreach": "@outputs('Output')?['caseFiles']",
                                                                                "actions": {
                                                                                    "Append_files": {
                                                                                        "type": "AppendToArrayVariable",
                                                                                        "inputs": {
                                                                                            "name": "files",
                                                                                            "value": {
                                                                                                "dateUploaded": "@{formatDateTime(addSeconds('1970-01-01T00:00:00Z', int(items('For_each-copy')?['dateUploaded'])), 'yyyy-MM-dd HH:mm:ss')}",
                                                                                                "originalFilename": "@{string(items('For_each-copy')?['originalFilename'])}",
                                                                                                "fileToken": "@{string(items('For_each-copy')?['fileToken'])}",
                                                                                                "isCaseScreenshot": "@{string(items('For_each-copy')?['isCaseScreenshot'])}"
                                                                                            }
                                                                                        }
                                                                                    }
                                                                                },
                                                                                "type": "Foreach"
                                                                            }
                                                                        }
                                                                    },
                                                                    "expression": {
                                                                        "and": [
                                                                            {
                                                                                "equals": [
                                                                                    "@outputs('Output')?['caseFiles']",
                                                                                    "@null"
                                                                                ]
                                                                            }
                                                                        ]
                                                                    },
                                                                    "type": "If"
                                                                }
                                                            }
                                                        },
                                                        "expression": {
                                                            "and": [
                                                                {
                                                                    "greater": [
                                                                        "@length(body('Run_query_and_list_results')?['value'])",
                                                                        0
                                                                    ]
                                                                }
                                                            ]
                                                        },
                                                        "type": "If"
                                                    }
                                                },
                                                "else": {
                                                    "actions": {}
                                                },
                                                "expression": {
                                                    "or": [
                                                        {
                                                            "equals": [
                                                                "@items('For_each_Incident')?['threatType']",
                                                                "Critical Threat"
                                                            ]
                                                        },
                                                        {
                                                            "equals": [
                                                                "@items('For_each_Incident')?['threatType']",
                                                                "High Threat"
                                                            ]
                                                        },
                                                        {
                                                            "equals": [
                                                                "@items('For_each_Incident')?['threatType']",
                                                                "Substantial Threat"
                                                            ]
                                                        }
                                                    ]
                                                },
                                                "type": "If"
                                            }
                                        },
                                        "type": "Foreach"
                                    }
                                }
                            },
                            "expression": {
                                "and": [
                                    {
                                        "equals": [
                                            "@body('HTTP_Incident')?['result']",
                                            "@null"
                                        ]
                                    },
                                    {
                                        "equals": [
                                            "",
                                            ""
                                        ]
                                    }
                                ]
                            },
                            "type": "If"
                        }

          },

          "outputs": {}
        },

        "parameters": {
          "$connections": {
            "value": {
              "azuremonitorlogs": {
                "connectionId": "[resourceId('Microsoft.Web/connections', variables('azureMonitorConnectionName'))]",
  "connectionName": "[variables('azureMonitorConnectionName')]",
  "id": "[concat('/subscriptions/', subscription().subscriptionId, '/providers/Microsoft.Web/locations/', resourceGroup().location, '/managedApis/azuremonitorlogs')]"
              },
              "azureloganalyticsdatacollector": {
                "connectionId": "[resourceId('Microsoft.Web/connections', variables('dataCollectorConnectionName'))]",
                "connectionName": "[variables('dataCollectorConnectionName')]",
                "id": "[concat('/subscriptions/', subscription().subscriptionId, '/providers/Microsoft.Web/locations/', resourceGroup().location, '/managedApis/azureloganalyticsdatacollector')]"
              }
            }
          }
        }
      }
    }
  ]

}
'''

print("PARAMETERISE LOGIC APP content defined as a string block.")

PARAMETERISE LOGIC APP content defined as a string block.


In [8]:
import json
import re

def clean_json_string(s):
    # Remove trailing commas from objects/arrays that often cause issues in manual pastes
    s = re.sub(r',\s*([}\]])', r'\1', s)
    return s

json_vars = [
    ('arm_schema_path', arm_schema_path),
    ('original_logic_app_path', original_logic_app_path),
    ('parameterise_logic_app_path', parameterise_logic_app_path),
    ('input_logic_app_content', input_logic_app_content)
]

parsed_data = {}
error_found = False

for name, content in json_vars:
    try:
        # Clean and load
        cleaned_content = clean_json_string(content)
        parsed_data[name] = json.loads(cleaned_content, strict=False)
        print(f"Successfully parsed: {name}")
    except json.JSONDecodeError as e:
        print(f"FAILED to parse {name}: {e}")
        error_found = True

# Map to expected variables for subsequent cells
arm_schema = parsed_data.get('arm_schema_path')
original_logic_app = parsed_data.get('original_logic_app_path')
parameterise_logic_app = parsed_data.get('parameterise_logic_app_path')
input_logic_app = parsed_data.get('input_logic_app_content')

working_object = {}
if not error_found:
    print("\nAll JSON content loaded. Empty working object initialized.")
else:
    print("\nSome blocks failed to parse. Please check for syntax errors in the failed blocks above.")

Successfully parsed: arm_schema_path
FAILED to parse original_logic_app_path: Expecting ',' delimiter: line 143 column 26 (char 7046)
FAILED to parse parameterise_logic_app_path: Expecting ',' delimiter: line 194 column 26 (char 7962)
Successfully parsed: input_logic_app_content

Some blocks failed to parse. Please check for syntax errors in the failed blocks above.


In [9]:
logic_app_workflow_resource = None

# Check if variable exists and is not None
if 'input_logic_app' in locals() and input_logic_app:
    if 'resources' in input_logic_app and isinstance(input_logic_app['resources'], list):
        for resource in input_logic_app['resources']:
            if resource.get('type') == 'Microsoft.Logic/workflows':
                logic_app_workflow_resource = resource
                break

if logic_app_workflow_resource:
    print("Successfully identified and extracted the 'Microsoft.Logic/workflows' resource.")
else:
    print("Error: 'Microsoft.Logic/workflows' resource not found. Ensure 'input_logic_app_content' is valid JSON.")

Successfully identified and extracted the 'Microsoft.Logic/workflows' resource.


In [10]:
import json

# SECTION 2: Create/Normalize Top-Level Shell

# Ensure working_object is initialized as an empty dictionary if it's not already
if 'working_object' not in locals():
    working_object = {}
    print("Warning: 'working_object' was not found, initializing as empty dictionary.")

# Set $schema and contentVersion from ARM_schema
# Using .get() for safety, though ARM_schema is expected to have these
working_object['$schema'] = arm_schema.get('$schema', 'https://schema.management.azure.com/schemas/2019-04-01/deploymentTemplate.json#')
working_object['contentVersion'] = arm_schema.get('contentVersion', '1.0.0.0')

# Set metadata from ARM_schema
# If ARM_schema has metadata, copy it. Otherwise, initialize as empty or default.
working_object['metadata'] = arm_schema.get('metadata', {
    "title": "Logic app Title",
    "description": "Logic app description",
    "prerequisites": "",
    "lastUpdateTime": "",
    "author": {}
})

# Ensure parameters, variables, and resources exist as empty dictionaries or lists
if 'parameters' not in working_object:
    working_object['parameters'] = {}
if 'variables' not in working_object:
    working_object['variables'] = {}
if 'resources' not in working_object:
    working_object['resources'] = []

print("Section 2: Top-level shell created/normalized successfully.")
# print(json.dumps(working_object, indent=2)) # Uncomment to inspect the working_object after this section

Section 2: Top-level shell created/normalized successfully.


In [11]:
import re

print("Section X: Identifying canonical API names and types...")

# 1. Initialize an empty dictionary named canonical_api_names_map
canonical_api_names_map = {}

# 2. Initialize another empty dictionary named canonical_api_details
canonical_api_details = {}

# 3. Access the connections_value dictionary
connections_value = logic_app_workflow_resource.get('properties', {}).get('parameters', {}).get('$connections', {}).get('value', {})

if connections_value:
    for connection_key, connection_details in connections_value.items():
        # 5. Extract its full id string
        connection_id = connection_details.get('id', '')

        api_type = None
        raw_api_name = None

        # 6. Determine the api_type
        if '/managedApis/' in connection_id:
            api_type = 'managedApis'
            raw_api_name = connection_id.split('/managedApis/')[-1]
        elif '/customApis/' in connection_id:
            api_type = 'customApis'
            raw_api_name = connection_id.split('/customApis/')[-1]
        elif 'customApis_' in connection_id:
            # Handle cases where it's already a parameter reference like [parameters('customApis_..._externalid')]
            api_type = 'customApis'
            raw_api_name = connection_key
        elif 'connections_' in connection_id:
            api_type = 'managedApis'
            raw_api_name = connection_key
        else:
            print(f"  Warning: Could not determine API type for connection ID: {connection_id}. Skipping consolidation for {connection_key}.")
            continue

        if raw_api_name and api_type:
            # 8. Clean the raw API name by removing numerical suffixes
            canonical_api_name = re.sub(r'-\d+$', '', raw_api_name)

            # 9. Add an entry to canonical_api_names_map
            canonical_api_names_map[connection_key] = canonical_api_name

            # 10. Add to canonical_api_details if not already present
            if canonical_api_name not in canonical_api_details:
                canonical_api_details[canonical_api_name] = {'type': api_type}
                print(f"  Identified canonical API '{canonical_api_name}' of type '{api_type}'.")

print("Section X: Canonical API names and types identified successfully.")

Section X: Identifying canonical API names and types...
  Identified canonical API 'azuremonitorlogs' of type 'managedApis'.
  Identified canonical API 'azureloganalyticsdatacollector' of type 'managedApis'.
Section X: Canonical API names and types identified successfully.


In [12]:
import json

# SECTION 3: Build Parameters Section

# Helper function to add a parameter if it doesn't already exist
def add_parameter(param_name, param_type, default_value):
    # Modified: Removed 'description' and 'display_name' from function signature
    # to ensure no 'metadata' blocks are generated, as per the task.
    if param_name not in working_object['parameters']:
        param_entry = {
            "type": param_type,
            "defaultValue": default_value,
        }
        # No metadata block is added here, fulfilling the requirement.
        working_object['parameters'][param_name] = param_entry
        print(f"  Added parameter: {param_name}")
    else:
        print(f"  Parameter {param_name} already exists, skipping.")

# Helper function to check for and extract Log Analytics workspace references
def extract_workspace_references(logic_app_resource):
    definition = logic_app_resource.get('properties', {}).get('definition', {})
    triggers = definition.get('triggers', {})
    actions = definition.get('actions', {})

    # Default values for extraction
    extracted_subscription_id = None
    extracted_resource_group = None
    extracted_workspace_name = None
    is_workspace_referenced = False

    # Helper to process paths
    def process_path(path):
        nonlocal extracted_subscription_id, extracted_resource_group, extracted_workspace_name, is_workspace_referenced
        if "/subscriptions/" in path and "/resourceGroups/" in path and "/workspaces/" in path:
            is_workspace_referenced = True
            try:
                # Attempt to extract values from the path
                sub_start = path.find("/subscriptions/") + len("/subscriptions/")
                sub_end = path.find("/", sub_start)
                extracted_subscription_id = path[sub_start:sub_end]

                rg_start = path.find("/resourceGroups/") + len("/resourceGroups/")
                rg_end = path.find("/", rg_start)
                extracted_resource_group = path[rg_start:rg_end]

                ws_start = path.find("/workspaces/") + len("/workspaces/")
                ws_end = path.find("/", ws_start) if path.find("/", ws_start) != -1 else len(path)
                extracted_workspace_name = path[ws_start:ws_end]
            except Exception as e:
                print(f"  Warning: Could not fully extract workspace details from path: {path}. Error: {e}")

    # Check triggers
    for trigger_name, trigger_details in triggers.items():
        if trigger_details.get('type') == 'ApiConnection' and 'inputs' in trigger_details:
            process_path(trigger_details['inputs'].get('path', ''))
            if is_workspace_referenced: break # Stop if found in triggers

    # Check actions if not found in triggers
    if not is_workspace_referenced:
        for action_name, action_details in actions.items():
            if action_details.get('type') == 'ApiConnection' and 'inputs' in action_details:
                process_path(action_details['inputs'].get('path', ''))
                if is_workspace_referenced: break # Stop if found in actions

    return {
        'is_referenced': is_workspace_referenced,
        'subscriptionId': extracted_subscription_id,
        'resourceGroup': extracted_resource_group,
        'workspaceName': extracted_workspace_name
    }

print("Section 3: Building parameters section...")

# Resolve the default logic app name for use in defaultValues as a NEUTRAL placeholder
default_logic_app_name_resolved = "logic-app-name"

# 1. Parameterize Logic App Name with neutral default and standardize to 'PlaybookName'
logic_app_name_param_for_ref = "PlaybookName" # Instruction 1: Always set to 'PlaybookName'

add_parameter(
    param_name=logic_app_name_param_for_ref,
    param_type="String", # Instruction 2: param_type to 'String'
    default_value=default_logic_app_name_resolved # Instruction 2: No description, no display_name to suppress metadata
)

# 2. Conditionally add parameters for Workspace details
workspace_refs = extract_workspace_references(logic_app_workflow_resource)

if workspace_refs['is_referenced']:
    print("  Log Analytics workspace references detected. Adding workspace parameters.")

    add_parameter(
        param_name="workspaceSubscriptionId",
        param_type="String", # Instruction 3: param_type to 'String'
        default_value="your-subscription-id" # Instruction 3: No description, no display_name
    )

    add_parameter(
        param_name="workspaceResourceGroup",
        param_type="String", # Instruction 3: param_type to 'String'
        default_value="your-resource-group" # Instruction 3: No description, no display_name
    )

    add_parameter(
        param_name="workspaceName",
        param_type="String", # Instruction 3: param_type to 'String'
        default_value="your-log-analytics-workspace-name" # Instruction 3: No description, no display_name
    )
else:
    print("  No Log Analytics workspace references detected. Skipping workspace parameters.")

# 3. Parameterize Connection External IDs AND Connection Names using canonical names
connections_value = logic_app_workflow_resource.get('properties', {}).get('parameters', {}).get('$connections', {}).get('value', {})

if connections_value:
    print("  Processing connection parameters...")
    generated_canonical_params = set() # To ensure parameters are generated only once per canonical API

    for connection_key, connection_details in connections_value.items():
        canonical_api_name = canonical_api_names_map.get(connection_key)
        if not canonical_api_name:
            print(f"  Skipping connection {connection_key} as canonical name could not be determined.")
            continue

        api_type = canonical_api_details.get(canonical_api_name, {}).get('type')
        if not api_type:
            print(f"  Skipping connection {connection_key} as API type for canonical '{canonical_api_name}' could not be determined.")
            continue

        # Generate parameters only once per canonical API name
        if canonical_api_name in generated_canonical_params:
            continue # Parameters already added for this canonical API

        # Instruction 4a: Dynamically define default_ext_id_value
        # Reintroducing external ID parameter for custom APIs
        if api_type == 'customApis':
            param_ext_id_name = f"connections_{canonical_api_name}_externalid"
            default_ext_id_value = f"/subscriptions/your-subscription-id/resourceGroups/your-resource-group/providers/Microsoft.Web/customApis/{canonical_api_name}"
            add_parameter(
                param_name=param_ext_id_name,
                param_type="String",
                default_value=default_ext_id_value
            )
            print(f"  Added custom API external ID parameter: {param_ext_id_name}")

        # Instruction 4b: Dynamically define default_conn_name_value
        default_conn_name_value = f"your-connection-name-{canonical_api_name}"
        # Parameter for Connection Name (now based on canonical API name)
        param_conn_name = f"connections_{canonical_api_name}_name"

        # The connection name parameter is no longer used, as connectionName is handled via variables in Section 11
        # Removing the add_parameter call for connections_*_name

        generated_canonical_params.add(canonical_api_name)
else:
    print("  No connections found to parameterize in 'properties.parameters.$connections.value'.")


print("Section 3: Parameters section built successfully.")
# print(json.dumps(working_object['parameters'], indent=2)) # Uncomment to inspect the parameters after this section

Section 3: Building parameters section...
  Added parameter: PlaybookName
  No Log Analytics workspace references detected. Skipping workspace parameters.
  Processing connection parameters...
Section 3: Parameters section built successfully.


In [13]:
import json

# SECTION 4: Build Variables Section

# Helper function to add a variable if it doesn't already exist
def add_variable(var_name, var_value):
    if var_name not in working_object['variables']:
        working_object['variables'][var_name] = var_value
        print(f"  Added variable: {var_name}")
    else:
        print(f"  Variable {var_name} already exists, skipping.")

print("Section 4: Building variables section...")

# 1. Define logicAppName variable if the logic app name was parameterized
# The parameter name for the Logic App is now standardized to 'PlaybookName'
logic_app_name_param_for_variable = "PlaybookName"

# Add logicAppName variable using the standardized PlaybookName parameter
add_variable(
    var_name="logicAppName",
    var_value=f"[parameters('{logic_app_name_param_for_variable}')]"
)

# 2. Add 'location' variable to reference the 'location' parameter
# Removed as per user request.
# add_variable(
#     var_name="location",
#     var_value="[parameters('location')]"
# )

# 3. Parameterize Connection Names for use as variables using canonical names
if canonical_api_details:
    print("  Processing connection name variables using canonical API names...")
    for canonical_api_name in canonical_api_details.keys():
        var_name = f"{canonical_api_name}ConnectionName"
        # Construct the variable value to follow the pattern [concat('<apiName>-', parameters('PlaybookName'))]
        var_value = f"[concat('{canonical_api_name}-', parameters('{logic_app_name_param_for_variable}'))]"
        add_variable(var_name, var_value)
else:
    print("  No canonical API details found to build connection name variables for.")

print("Section 4: Variables section built successfully.")
# print(json.dumps(working_object['variables'], indent=2)) # Uncomment to inspect the variables after this section

Section 4: Building variables section...
  Added variable: logicAppName
  Processing connection name variables using canonical API names...
  Added variable: azuremonitorlogsConnectionName
  Added variable: azureloganalyticsdatacollectorConnectionName
Section 4: Variables section built successfully.


In [14]:
import json

# SECTION 5: Build Microsoft.Web/connections resources

print("Section 5: Building Microsoft.Web/connections resources...")

# Clear existing resources before rebuilding
working_object['resources'] = []

if canonical_api_details:
    for canonical_api_name, details in canonical_api_details.items():
        api_type = details.get('type')

        if not api_type:
            print(f"  Warning: No API type found for canonical API '{canonical_api_name}'. Skipping.")
            continue

        connection_name_reference = f"[variables('{canonical_api_name}ConnectionName')]"

        # Dynamically build the API ID based on whether it is a managed or custom API
        # Managed APIs use the location-based path
        if api_type == 'managedApis':
            api_id_expression = f"[concat('/subscriptions/', subscription().subscriptionId, '/providers/Microsoft.Web/locations/', resourceGroup().location, '/managedApis/{canonical_api_name}')]"
        # Custom APIs should now reference the externalid parameter
        else: # api_type == 'customApis'
            # Instruction for subtask: reference connections_{canonical_api_name}_externalid parameter
            param_ext_id_name = f"connections_{canonical_api_name}_externalid"
            api_id_expression = f"[parameters('{param_ext_id_name}')]"

        connection_resource = {
            "type": "Microsoft.Web/connections",
            "apiVersion": "2016-06-01",
            "name": connection_name_reference,
            "location": "[resourceGroup().location]",
            "properties": {
                "api": {
                    "id": api_id_expression
                },
                "displayName": canonical_api_name.title()
            }
        }

        working_object['resources'].append(connection_resource)
        print(f"  Added consolidated connection resource for: {canonical_api_name} ({api_type})")

else:
    print("  No canonical API details found to build Microsoft.Web/connections resources for.")

print("Section 5: Microsoft.Web/connections resources built successfully with corrected API IDs.")
# print(json.dumps(working_object['resources'], indent=2)) # Uncomment to inspect the working_object after this section


Section 5: Building Microsoft.Web/connections resources...
  Added consolidated connection resource for: azuremonitorlogs (managedApis)
  Added consolidated connection resource for: azureloganalyticsdatacollector (managedApis)
Section 5: Microsoft.Web/connections resources built successfully with corrected API IDs.


In [15]:
import json

# SECTION 6: Build Microsoft.Logic/workflows resource shell

print("Section 6: Building Microsoft.Logic/workflows resource shell...")

# Prepare the dependsOn array
depends_on_connections = []
for resource in working_object['resources']:
    if resource.get('type') == 'Microsoft.Web/connections':
        # Extract the connection name expression, e.g., "[variables('azuresentinelConnectionName')]"
        connection_name_expr = resource.get('name')
        if connection_name_expr:
            # Construct the full resourceId expression for the dependency
            # Corrected: Strip the outer square brackets from connection_name_expr
            depends_on_connections.append(f"[resourceId('Microsoft.Web/connections', {connection_name_expr.strip('[]')})]")

# Get logic app name from parameters
# The logic app name parameter is now standardized to 'PlaybookName'
logic_app_name_param_value = "[parameters('PlaybookName')]"

# Build the main Microsoft.Logic/workflows resource
logic_app_resource = {
    "type": "Microsoft.Logic/workflows",
    "apiVersion": logic_app_workflow_resource.get('apiVersion', '2017-07-01'), # Default API version if not present
    "name": logic_app_name_param_value,
    "location": "[resourceGroup().location]", # Using resource group location directly
    "identity": logic_app_workflow_resource.get('identity', {'type': 'SystemAssigned'}), # Keep existing identity or default
    "properties": {}, # Initialize properties as an empty dictionary
    "dependsOn": depends_on_connections
}

# Remove any existing Microsoft.Logic/workflows resource before adding the new one
working_object['resources'] = [res for res in working_object['resources'] if res.get('type') != 'Microsoft.Logic/workflows']

# Add the logic app resource to the working object's resources
working_object['resources'].append(logic_app_resource)
print("Section 6: Microsoft.Logic/workflows resource shell built successfully.")
# print(json.dumps(working_object['resources'], indent=2)) # Uncomment to inspect the resources after this section

Section 6: Building Microsoft.Logic/workflows resource shell...
Section 6: Microsoft.Logic/workflows resource shell built successfully.


In [16]:
import json

# SECTION 7: Build workflow definition.parameters

print("Section 7: Building workflow definition.parameters...")

# Helper function to ensure the Microsoft.Logic/workflows resource is in working_object['resources']
def _ensure_logic_app_workflow_resource(working_object, logic_app_workflow_resource):
    logic_app_res = None
    for resource in working_object['resources']:
        if resource.get('type') == 'Microsoft.Logic/workflows':
            logic_app_res = resource
            break

    if logic_app_res:
        return logic_app_res
    else:
        print("  Microsoft.Logic/workflows resource not found in working_object. Rebuilding it now (from Section 6 logic).")
        # Re-run Section 6 logic to build and add the resource
        depends_on_connections = []
        for resource in working_object['resources']: # Now working_object['resources'] only has Microsoft.Web/connections
            if resource.get('type') == 'Microsoft.Web/connections':
                connection_name_expr = resource.get('name')
                if connection_name_expr:
                    depends_on_connections.append(f"[resourceId('Microsoft.Web/connections', {connection_name_expr.strip('[]')})]")

        logic_app_name_param_value = None
        for param_name, param_details in working_object.get('parameters', {}).items():
            if param_details.get('metadata', {}).get('displayName') == 'Logic App Name':
                logic_app_name_param_value = f"[parameters('{param_name}')]"
                break
        # Fallback if logic app name parameter not found, use original resource name or a generic parameter
        if not logic_app_name_param_value:
            logic_app_name_param_value = logic_app_workflow_resource.get('name', "[parameters('logicAppName')]")

        # Build the main Microsoft.Logic/workflows resource
        logic_app_res = {
            "type": "Microsoft.Logic/workflows",
            "apiVersion": logic_app_workflow_resource.get('apiVersion', '2017-07-01'), # Default API version if not present
            "name": logic_app_name_param_value,
            "location": "[resourceGroup().location]", # Using resource group location directly
            "identity": logic_app_workflow_resource.get('identity', {'type': 'SystemAssigned'}), # Keep existing identity or default
            "properties": {}, # Initialize properties as an empty dictionary
            "dependsOn": depends_on_connections
        }

        working_object['resources'].append(logic_app_res)
        print("  Microsoft.Logic/workflows resource rebuilt and added.")
        return logic_app_res


# Find or rebuild the Microsoft.Logic/workflows resource in the working_object
logic_app_resource_in_working_object = _ensure_logic_app_workflow_resource(working_object, logic_app_workflow_resource)

if not logic_app_resource_in_working_object:
    raise ValueError("Section 7 Error: Microsoft.Logic/workflows resource not found in working_object.")

# Ensure properties key exists
if 'properties' not in logic_app_resource_in_working_object:
    logic_app_resource_in_working_object['properties'] = {}

# Set the definition with required schema and contentVersion
definition = {
    "$schema": "https://schema.management.azure.com/providers/Microsoft.Logic/schemas/2016-06-01/workflowdefinition.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {}
}
logic_app_resource_in_working_object['properties']['definition'] = definition

# Extract original parameters from the input logic app's definition
original_definition_parameters = logic_app_workflow_resource.get('properties', {}).get('definition', {}).get('parameters', {})

new_definition_parameters = {}

for param_name, param_details in original_definition_parameters.items():
    new_param_details = param_details.copy()

    # Handle $connections parameters within the definition
    if param_name.startswith('$connections') and isinstance(new_param_details.get('defaultValue'), dict):
        default_value = new_param_details['defaultValue'].get('value', {})
        if isinstance(default_value, dict) and 'connection' in default_value:
            connection_id = default_value['connection'].get('id')
            if connection_id:
                raw_api_name_match = None
                if '/managedApis/' in connection_id:
                    raw_api_name_match = connection_id.split('/managedApis/')[-1]
                elif '/customApis/' in connection_id:
                    raw_api_name_match = connection_id.split('/customApis/')[-1]

                if raw_api_name_match:
                    canonical_api_name = canonical_api_names_map.get(raw_api_name_match)
                    if canonical_api_name:
                        connection_variable_name = f"{canonical_api_name}ConnectionName"
                        new_param_details['defaultValue']['value']['connection']['id'] = f"[resourceId('Microsoft.Web/connections', variables('{connection_variable_name}'))]"
                        print(f"  Parameterized internal connection ID for '{param_name}' using canonical '{canonical_api_name}'.")

    new_definition_parameters[param_name] = new_param_details

# Assign the new parameters to the workflow definition
logic_app_resource_in_working_object['properties']['definition']['parameters'] = new_definition_parameters

print("Section 7: Workflow definition built with $schema, contentVersion and parameters successfully.")

Section 7: Building workflow definition.parameters...
Section 7: Workflow definition built with $schema, contentVersion and parameters successfully.


In [17]:
import json
import re

# SECTION 8: Build workflow definition.triggers

print("Section 8: Building workflow definition.triggers with fixed connection names...")

logic_app_resource_in_working_object = next((res for res in working_object['resources'] if res.get('type') == 'Microsoft.Logic/workflows'), None)

original_triggers = logic_app_workflow_resource.get('properties', {}).get('definition', {}).get('triggers', {})
new_triggers_str = json.dumps(original_triggers)

# 1. Standardize PlaybookName parameter reference
old_param_pattern = r"parameters\('workflows_AEAZ_Firewall_IP_Blocking_name'\)"
new_triggers_str = re.sub(old_param_pattern, "parameters('PlaybookName')", new_triggers_str)

# 2. Standardize Connection names within inputs.host.connection.name
# Find patterns like @parameters('$connections')['azuresentinel']['connectionId']
# and map them to variables('azuresentinelConnectionName')
def replace_connection_expr(match):
    conn_key = match.group(1)
    # Check if we have a canonical mapping for this key
    canonical_name = canonical_api_names_map.get(conn_key, conn_key)
    return f"@parameters('$connections')['{conn_key}']['connectionId']"

# Actually, for triggers and actions to work with the consolidated parameter structure,
# the host.connection.name should reference the key in the workflow's parameters.$connections
# No change is strictly needed here if Section 11 is consistent with the trigger keys.
# However, the error 'azuremonitorlogs-test_5_deployemnt_template' suggests a mismatch in Section 11 naming.

new_triggers = json.loads(new_triggers_str)

# Apply path replacements for Workspace
replacements = {
    "/subscriptions/05482007-a538-44e4-8e37-7cab6b74255e": "/subscriptions/@{encodeURIComponent(parameters('workspaceSubscriptionId'))}",
    "/resourceGroups/AE_AZURE_SE_RG": "/resourceGroups/@{encodeURIComponent(parameters('workspaceResourceGroup'))}",
    "/workspaces/AE-AZ-AzureSentinel": "/workspaces/@{encodeURIComponent(parameters('workspaceName'))}"
}

for trigger_name, trigger_details in new_triggers.items():
    if trigger_details.get('type') == 'ApiConnection' or trigger_details.get('type') == 'ApiConnectionWebhook':
        if 'inputs' in trigger_details and 'path' in trigger_details['inputs']:
            modified_path = trigger_details['inputs']['path']
            for old_value, new_value in replacements.items():
                modified_path = modified_path.replace(old_value, new_value)
            trigger_details['inputs']['path'] = modified_path

logic_app_resource_in_working_object['properties']['definition']['triggers'] = new_triggers
print("Section 8: Triggers built successfully.")

Section 8: Building workflow definition.triggers with fixed connection names...
Section 8: Triggers built successfully.


In [18]:
import json
import re

# SECTION 9: Build workflow definition.actions

print("Section 9: Building workflow definition.actions with fixed connection names...")

logic_app_resource_in_working_object = next((res for res in working_object['resources'] if res.get('type') == 'Microsoft.Logic/workflows'), None)

original_actions = logic_app_workflow_resource.get('properties', {}).get('definition', {}).get('actions', {})
new_actions_str = json.dumps(original_actions)

# 1. Standardize PlaybookName parameter reference
old_param_pattern = r"parameters\('workflows_AEAZ_Firewall_IP_Blocking_name'\)"
new_actions_str = re.sub(old_param_pattern, "parameters('PlaybookName')", new_actions_str)

new_actions = json.loads(new_actions_str)

# Apply path replacements for Workspace
replacements = {
    "/subscriptions/05482007-a538-44e4-8e37-7cab6b74255e": "/subscriptions/@{encodeURIComponent(parameters('workspaceSubscriptionId'))}",
    "/resourceGroups/AE_AZURE_SE_RG": "/resourceGroups/@{encodeURIComponent(parameters('workspaceResourceGroup'))}",
    "/workspaces/AE-AZ-AzureSentinel": "/workspaces/@{encodeURIComponent(parameters('workspaceName'))}"
}

for action_name, action_details in new_actions.items():
    if action_details.get('type') == 'ApiConnection':
        if 'inputs' in action_details and 'path' in action_details['inputs']:
            modified_path = action_details['inputs']['path']
            for old_value, new_value in replacements.items():
                modified_path = modified_path.replace(old_value, new_value)
            action_details['inputs']['path'] = modified_path

logic_app_resource_in_working_object['properties']['definition']['actions'] = new_actions
print("Section 9: Actions built successfully.")

Section 9: Building workflow definition.actions with fixed connection names...
Section 9: Actions built successfully.


In [19]:
import json

# SECTION 10: Build workflow definition.outputs

print("Section 10: Building workflow definition.outputs...")

# Find the Microsoft.Logic/workflows resource in the working_object
logic_app_resource_in_working_object = None
for resource in working_object['resources']:
    if resource.get('type') == 'Microsoft.Logic/workflows':
        logic_app_resource_in_working_object = resource
        break

if not logic_app_resource_in_working_object:
    raise ValueError("Section 10 Error: Microsoft.Logic/workflows resource not found in working_object.")

# Ensure properties and definition keys exist
if 'properties' not in logic_app_resource_in_working_object:
    logic_app_resource_in_working_object['properties'] = {}
if 'definition' not in logic_app_resource_in_working_object['properties']:
    logic_app_resource_in_working_object['properties']['definition'] = {}

# Extract original outputs from the input logic app's definition
original_outputs = logic_app_workflow_resource.get('properties', {}).get('definition', {}).get('outputs', {})

# Assign the outputs to the workflow definition
logic_app_resource_in_working_object['properties']['definition']['outputs'] = original_outputs

print("Section 10: Workflow definition.outputs built successfully.")
# print(json.dumps(logic_app_resource_in_working_object['properties']['definition']['outputs'], indent=2)) # Uncomment to inspect


Section 10: Building workflow definition.outputs...
Section 10: Workflow definition.outputs built successfully.


In [20]:
import json

# SECTION 11: Build properties.parameters.$connections.value

print("Section 11: Building properties.parameters.$connections.value...")

# Helper function to ensure the Microsoft.Logic/workflows resource is in working_object['resources']
def _ensure_logic_app_workflow_resource(working_object, logic_app_workflow_resource):
    logic_app_res = None
    for resource in working_object['resources']:
        if resource.get('type') == 'Microsoft.Logic/workflows':
            logic_app_res = resource
            break

    if logic_app_res:
        return logic_app_res
    else:
        print("  Rebuilding Logic App workflow resource shell...")
        depends_on_connections = []
        for resource in working_object['resources']:
            if resource.get('type') == 'Microsoft.Web/connections':
                name_expr = resource.get('name')
                if name_expr:
                    depends_on_connections.append(f"[resourceId('Microsoft.Web/connections', {name_expr.strip('[]')})]")

        logic_app_res = {
            "type": "Microsoft.Logic/workflows",
            "apiVersion": "2017-07-01",
            "name": "[parameters('PlaybookName')]",
            "location": "[resourceGroup().location]",
            "identity": {"type": "SystemAssigned"},
            "properties": {},
            "dependsOn": depends_on_connections
        }
        working_object['resources'].append(logic_app_res)
        return logic_app_res

logic_app_resource_in_working_object = _ensure_logic_app_workflow_resource(working_object, logic_app_workflow_resource)

if 'properties' not in logic_app_resource_in_working_object:
    logic_app_resource_in_working_object['properties'] = {}
if 'parameters' not in logic_app_resource_in_working_object['properties']:
    logic_app_resource_in_working_object['properties']['parameters'] = {}
if '$connections' not in logic_app_resource_in_working_object['properties']['parameters']:
    logic_app_resource_in_working_object['properties']['parameters']['$connections'] = {}

original_connections_value = logic_app_workflow_resource.get('properties', {}).get('parameters', {}).get('$connections', {}).get('value', {})
new_connections_value = json.loads(json.dumps(original_connections_value))

connection_key_to_canonical_map = {k: v for k, v in canonical_api_names_map.items()}

for connection_key in list(new_connections_value.keys()):
    canonical_api_name = connection_key_to_canonical_map.get(connection_key)
    if not canonical_api_name:
        new_connections_value.pop(connection_key, None)
        continue

    api_type = canonical_api_details.get(canonical_api_name, {}).get('type')
    if not api_type:
        new_connections_value.pop(connection_key, None)
        continue

    # Update the internal 'id' using the correct API type logic
    if api_type == 'managedApis':
        new_connections_value[connection_key]['id'] = f"[concat('/subscriptions/', subscription().subscriptionId, '/providers/Microsoft.Web/locations/', resourceGroup().location, '/managedApis/{canonical_api_name}')]"
    else:
        # For customApis, reference the connections_{canonical_api_name}_externalid parameter
        param_ext_id_name = f"connections_{canonical_api_name}_externalid"
        new_connections_value[connection_key]['id'] = f"[parameters('{param_ext_id_name}')]"

    connection_variable_name = f"{canonical_api_name}ConnectionName"
    new_connections_value[connection_key]['connectionId'] = f"[resourceId('Microsoft.Web/connections', variables('{connection_variable_name}'))]"
    new_connections_value[connection_key]['connectionName'] = f"[variables('{connection_variable_name}')]"

    # ENSURE connectionProperties IS ALWAYS PRESENT AS AN EMPTY OBJECT
    new_connections_value[connection_key]['connectionProperties'] = {}

logic_app_resource_in_working_object['properties']['parameters']['$connections']['value'] = new_connections_value
print("Section 11: Updated with correct API types for internal connection mappings (External ID parameter referenced). 'connectionProperties' are preserved and ensured for all entries.")

print("Generated $connections.value content:")
print(json.dumps(logic_app_resource_in_working_object['properties']['parameters']['$connections']['value'], indent=2))

Section 11: Building properties.parameters.$connections.value...
Section 11: Updated with correct API types for internal connection mappings (External ID parameter referenced). 'connectionProperties' are preserved and ensured for all entries.
Generated $connections.value content:
{
  "azuremonitorlogs-1": {
    "id": "[concat('/subscriptions/', subscription().subscriptionId, '/providers/Microsoft.Web/locations/', resourceGroup().location, '/managedApis/azuremonitorlogs')]",
    "connectionId": "[resourceId('Microsoft.Web/connections', variables('azuremonitorlogsConnectionName'))]",
    "connectionName": "[variables('azuremonitorlogsConnectionName')]",
    "connectionProperties": {}
  },
  "azureloganalyticsdatacollector": {
    "id": "[concat('/subscriptions/', subscription().subscriptionId, '/providers/Microsoft.Web/locations/', resourceGroup().location, '/managedApis/azureloganalyticsdatacollector')]",
    "connectionId": "[resourceId('Microsoft.Web/connections', variables('azureloga

In [21]:
import json
import re

# SECTION 12: Secret Parameterization
print("Section 12: Detecting and parameterizing secrets...")

# Find the Microsoft.Logic/workflows resource in the working_object
logic_app_resource_in_working_object = None
for resource in working_object['resources']:
    if resource.get('type') == 'Microsoft.Logic/workflows':
        logic_app_resource_in_working_object = resource
        break

if not logic_app_resource_in_working_object:
    raise ValueError("Section 12 Error: Microsoft.Logic/workflows resource not found in working_object.")

# Ensure properties and definition keys exist
if 'properties' not in logic_app_resource_in_working_object:
    logic_app_resource_in_working_object['properties'] = {}
if 'definition' not in logic_app_resource_in_working_object['properties']:
    logic_app_resource_in_working_object['properties']['definition'] = {}

workflow_definition = logic_app_resource_in_working_object['properties']['definition']

# Initialize tracking variables
secret_parameters = {}
secret_replacements = []
parameters_created_count = 0

# Helper functions
def contains_expression(value):
    if isinstance(value, str):
        # Regex to detect common Logic Apps expressions
        return re.search(r'@{.*}|@parameters\(.*\)|@variables\(.*\)|@body\(.*\)|@outputs\(.*\)|@triggerBody\(.*\)|@actions\(.*\)|@items\(.*\)|@concat\(.*\)', value)
    return False

def is_secret_field(key, parent_key=None):
    key_lower = str(key).lower()
    # parent_key_lower = str(parent_key).lower() if parent_key else '' # Not strictly needed with new logic

    # Rule 1: Allowlist of direct credential field names
    # Rule 3 & 6: Only parameterize fields whose names match credential patterns.
    credential_keys_allowlist = {
        'apikey', 'x-api-key', 'secretkey', 'clientsecret', 'password', 'sharedkey',
        'primarykey', 'secondarykey', 'connectionkey', 'authorization', 'token', 'accesstoken', 'bearertoken'
    }

    # If the key itself is in the credential allowlist, it's a secret.
    if key_lower in credential_keys_allowlist:
        return True

    # Rule 2: Explicitly do NOT parameterize generic headers or config values.
    # This is implicitly handled by the allowlist approach. If it's not in allowlist, it's not a secret.
    # No need for explicit denylist here as allowlist takes precedence.

    return False # If none of the specific conditions are met, it's not a secret.

def sanitize_param_name(original_name, path_elements, current_key=None):
    # Combine parts of the path for uniqueness, but keep it readable
    if current_key and not original_name.lower().endswith(str(current_key).lower()):
        name_parts = [p.replace('-', '_').replace('$', '') for p in path_elements if p and isinstance(p, str)]
        if name_parts:
            base_name = '_'.join(name_parts) + '_' + original_name.replace('-', '_').replace('$', '')
        else:
            base_name = original_name.replace('-', '_').replace('$', '')
    else:
        base_name = original_name.replace('-', '_').replace('$', '')

    # Remove non-alphanumeric characters and ensure it starts with a letter
    base_name = re.sub(r'[^a-zA-Z0-9_]', '', base_name)
    if not base_name[0].isalpha():
        base_name = 'param_' + base_name

    return base_name

def add_secure_parameter(param_name, default_value, description):
    global parameters_created_count
    if param_name not in working_object['parameters']:
        working_object['parameters'][param_name] = {
            "type": "securestring",
            "defaultValue": default_value,
            "metadata": {
                "description": description
            }
        }
        parameters_created_count += 1
        print(f"  Created securestring parameter: {param_name}")
    return param_name

def traverse_and_parameterize(obj, path_elements, parent_key=None):
    if isinstance(obj, dict):
        for key, value in obj.items():
            current_path_elements = path_elements + [key]
            if isinstance(value, (dict, list)):
                traverse_and_parameterize(value, current_path_elements, key)
            elif isinstance(value, str) and value.strip(): # Only consider non-empty strings
                # Rule 4: Never parameterize dynamic expressions
                if not contains_expression(value) and is_secret_field(key, parent_key):
                    # This is a candidate for parameterization
                    # Create a unique key for deduplication based on value and contextual field
                    dedup_key = (value, key)

                    if dedup_key in secret_parameters:
                        # Reuse existing parameter
                        param_name = secret_parameters[dedup_key]
                        replacement_expr = f"[parameters('{param_name}')]"
                        obj[key] = replacement_expr
                        secret_replacements.append({
                            'path': '.'.join(map(str, current_path_elements)),
                            'original_value_snippet': value[:50] + ('...' if len(value) > 50 else ''),
                            'parameter_name': param_name
                        })
                        print(f"  Reused parameter '{param_name}' for secret at {'.'.join(map(str, current_path_elements))}")
                    else:
                        # Create new parameter
                        base_param_name = sanitize_param_name(key, path_elements, key)
                        unique_param_name = base_param_name
                        i = 1
                        while unique_param_name in working_object['parameters']:
                            unique_param_name = f"{base_param_name}{i}"
                            i += 1

                        description = f"Secret for '{key}' in Logic App workflow."
                        add_secure_parameter(unique_param_name, value, description)

                        replacement_expr = f"[parameters('{unique_param_name}')]"
                        obj[key] = replacement_expr
                        secret_parameters[dedup_key] = unique_param_name
                        secret_replacements.append({
                            'path': '.'.join(map(str, current_path_elements)),
                            'original_value_snippet': value[:50] + ('...' if len(value) > 50 else ''),
                            'parameter_name': unique_param_name
                        })
                        print(f"  Parameterized secret at {'.'.join(map(str, current_path_elements))} with '{unique_param_name}'")
                # For non-secret strings or expressions, leave them as they are.

    elif isinstance(obj, list):
        for index, item in enumerate(obj):
            current_path_elements = path_elements + [index]
            if isinstance(item, (dict, list)):
                traverse_and_parameterize(item, current_path_elements, parent_key)

# Start traversal from the workflow definition's triggers and actions
print("  Traversing triggers...")
traverse_and_parameterize(workflow_definition.get('triggers', {}), ['properties', 'definition', 'triggers'])
print("  Traversing actions...")
traverse_and_parameterize(workflow_definition.get('actions', {}), ['properties', 'definition', 'actions'])

print("Section 12: Secret parameterization complete.")
print("\n--- Secret Parameterization Summary ---")
print(f"Total secrets detected: {len(secret_replacements)}")
print(f"Unique securestring parameters created: {parameters_created_count}")

if parameters_created_count > 0:
    print("Generated securestring parameters:")
    for param_name in sorted(list(set(secret_parameters.values()))):
        print(f"  - {param_name}")

if secret_replacements:
    print("Replacements occurred at JSON paths:")
    for rep in secret_replacements:
        print(f"  - Path: {rep['path']}")
        print(f"    Original Value Snippet: '{rep['original_value_snippet']}'")
        print(f"    Replaced With Parameter: '{rep['parameter_name']}'")
else:
    print("No hardcoded secrets were detected and parameterized.")

print("---------------------------------------")

Section 12: Detecting and parameterizing secrets...
  Traversing triggers...
  Traversing actions...
  Created securestring parameter: apiKey
  Parameterized secret at properties.definition.actions.HTTP_Auth.inputs.body.apiKey with 'apiKey'
  Created securestring parameter: secretKey
  Parameterized secret at properties.definition.actions.HTTP_Auth.inputs.body.secretKey with 'secretKey'
Section 12: Secret parameterization complete.

--- Secret Parameterization Summary ---
Total secrets detected: 2
Unique securestring parameters created: 2
Generated securestring parameters:
  - apiKey
  - secretKey
Replacements occurred at JSON paths:
  - Path: properties.definition.actions.HTTP_Auth.inputs.body.apiKey
    Original Value Snippet: '93171A4A-1636-4295-9508-7619A3C77C30'
    Replaced With Parameter: 'apiKey'
  - Path: properties.definition.actions.HTTP_Auth.inputs.body.secretKey
    Original Value Snippet: '1D5C1619-10BD-4474-B649-128E1A0CE9D450EE17BA-BDEC-...'
    Replaced With Parameter:

In [22]:
import json
import os

# SECTION END: Final Validation and Saving Output

print("Section END: Performing final validation and saving output...")

# --- Final Validation (Basic Check) ---
validation_errors = []
if not working_object:
    validation_errors.append("Working object is empty.")
else:
    required_keys = ['$schema', 'contentVersion', 'parameters', 'variables', 'resources']
    for key in required_keys:
        if key not in working_object:
            validation_errors.append(f"Missing top-level key: '{key}'.")

    if not isinstance(working_object.get('resources'), list) or not working_object['resources']:
        validation_errors.append("Resources section is missing or empty.")

    # Check for the main Logic App resource
    logic_app_resource_found = False
    for res in working_object.get('resources', []):
        if res.get('type') == 'Microsoft.Logic/workflows':
            logic_app_resource_found = True
            break

    if not logic_app_resource_found:
        validation_errors.append("Main 'Microsoft.Logic/workflows' resource not found in output.")

if validation_errors:
    print("Final Validation Warnings/Errors:")
    for error in validation_errors:
        print(f"  - {error}")
    print("Output may be malformed. Saving anyway...")
else:
    print("Basic validation passed: working_object appears well-formed.")

# --- Save Output JSON File ---
output_path = globals().get("output_path", "/content/output_logic_app.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(working_object, f, indent=2, ensure_ascii=False)

print(f"Saved output to: {output_path}")
print("All sections are finished.")

Section END: Performing final validation and saving output...
Basic validation passed: working_object appears well-formed.
Saved output to: /content/output_logic_app.json
All sections are finished.


In [23]:
# Validation for Section 11
print("Performing final validation of properties.parameters.$connections.value...")

logic_app_resource_in_working_object = next((res for res in working_object['resources'] if res.get('type') == 'Microsoft.Logic/workflows'), None)

if logic_app_resource_in_working_object:
    connections_value_to_validate = logic_app_resource_in_working_object.get('properties', {}).get('parameters', {}).get('$connections', {}).get('value', {})

    if not connections_value_to_validate:
        print("  Warning: No connections found in properties.parameters.$connections.value to validate.")
    else:
        for conn_key, conn_details in connections_value_to_validate.items():
            if 'id' not in conn_details:
                raise ValueError(f"Validation Error: Connection '{conn_key}' in properties.parameters.$connections.value is missing the required 'id' property.")
            if 'connectionId' not in conn_details:
                raise ValueError(f"Validation Error: Connection '{conn_key}' in properties.parameters.$connections.value is missing the required 'connectionId' property.")
            if 'connectionName' not in conn_details:
                raise ValueError(f"Validation Error: Connection '{conn_key}' in properties.parameters.$connections.value is missing the required 'connectionName' property.")
        print("  Validation successful: All connections in properties.parameters.$connections.value have 'id', 'connectionId', and 'connectionName' properties.")
else:
    print("  Microsoft.Logic/workflows resource not found in working_object for connection validation.")

Performing final validation of properties.parameters.$connections.value...
  Validation successful: All connections in properties.parameters.$connections.value have 'id', 'connectionId', and 'connectionName' properties.
